In [142]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn import set_config

In [69]:
df = pd.read_csv('/Users/tusharshukla/Machine-Learning-Roadmap/Level - 3/06. Working with Pipelines/Demo without using Pipelines/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [70]:
# Only using required columns
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
print("- Data after removing not required columns:")
display(df.head())
df.shape

- Data after removing not required columns:


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


(891, 8)

In [71]:
categorical_cols = df.select_dtypes(include=['object']).columns
categorical_cols

Index(['Sex', 'Embarked'], dtype='object')

In [75]:
print("Total Missing Values:")
print('-'*22)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

print("Total Missing Value(%):")
print('-'*22)
missing_percent = df.isnull().mean()*100
print(missing_percent[missing_percent > 0].round(2))

Total Missing Values:
----------------------
Age         177
Embarked      2
dtype: int64
Total Missing Value(%):
----------------------
Age         19.87
Embarked     0.22
dtype: float64


In [82]:
df.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [73]:
# Separeting Input & Output cols
X = df.drop(columns=['Survived'])
y =  df['Survived']

In [114]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)

print("Traning Set:")
display(X_train.head())
print(X_train.shape)

Traning Set:


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


(712, 7)


In [94]:
print('-'*22)
print(f"Traning Samples: {X_train.shape[0]}")
print(f"Testing Samples: {X_test.shape[0]}")
train_set_ratio = len(X_train)/len(df)
test_set_ratio = len(X_test)/len(df)
print('-'*22)
print(f"Training Ratio: {train_set_ratio:.2f}%")
print(f"Testing Ratio: {test_set_ratio:.2f}%")

----------------------
Traning Samples: 712
Testing Samples: 179
----------------------
Training Ratio: 0.80%
Testing Ratio: 0.20%


In [140]:
imputer = SimpleImputer() # imputing missing values
ohe = OneHotEncoder(sparse_output=False, drop='first') # encoding categorical cols into numerical also handling multicolinearity - drop='first'
scaler = MinMaxScaler() # Use minmax scaler when working with feature selection
dt_model = DecisionTreeClassifier(random_state=42)

In [135]:
# Imputing missing values - 1st step in pipeline
trf1 = ColumnTransformer([
    ('missing_value_imputer_for_age', imputer, [1]),
    ('missing_value_imputer_for_embarked', imputer, [-1])
], remainder='passthrough')

In [136]:
# Encoding categorical cols - 2nd step in pipeline
trf2 = ColumnTransformer([
    ('encode_sex_and_embarked', ohe, [1, -1])
], remainder='passthrough')

In [137]:
# Feature scaling - 3rd step in pipeline
trf3 = ColumnTransformer([
    ('feature_scaler', scaler, slice(0,9))
])

In [139]:
# Feature selection - 4th step in pipeline
trf4 = SelectKBest(score_func=chi2, k=5)

In [141]:
# Traing the model
trf5 = dt_model

In [143]:
set_config(display='diagram') # Visualize the pipeline

In [144]:
pipeline = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),
    ('trf3', trf3),
    ('trf4', trf4),
    ('trf5', trf5)
])

In [134]:
display(X_train['Embarked'].value_counts())
print('-'*30)
print(f"Total # of unique values: {X_train['Embarked'].nunique()}")

Embarked
S    525
C    125
Q     60
Name: count, dtype: int64

------------------------------
Total # of unique values: 3
